In [ ]:
import oracledb
from google.cloud.bigquery import Client
import os, json
from google.cloud import secretmanager
import pandas as pd

In [ ]:
def set_secrets_as_envs():
  secrets = secretmanager.SecretManagerServiceClient()
  resource_name = f"{os.environ['KNADA_TEAM_SECRET']}/versions/latest"
  secret = secrets.access_secret_version(name=resource_name)
  secret_str = secret.payload.data.decode('UTF-8')
  secrets = json.loads(secret_str)
  os.environ.update(secrets)

In [ ]:
def oracle_secrets():
  set_secrets_as_envs()
  return dict(
    user= os.getenv('DB_USER'),
    password= os.getenv('DB_PASSWORD'),
    host = os.getenv('DBT_ORCL_HOST'),
    service = os.getenv('DBT_ORCL_SERVICE'),
    project_id = os.getenv('BS_PROJECT_ID'),
    table_uri = os.getenv('BS_TABLE_URI'),  
    encoding="UTF-8",
    nencoding="UTF-8"
  )

oracle_secrets = oracle_secrets()

In [ ]:
def get_data_from_table():
    PROJECT_ID = oracle_secrets['project_id'] 
    TABLE_URI = oracle_secrets['table_uri'] 

    client = Client(project=PROJECT_ID)
    job = client.query(f"SELECT * FROM `{TABLE_URI}`")
       
    df = job.to_dataframe().drop_duplicates()
        
    return df

In [ ]:
def send_data():

    df = get_data_from_table()

    timestamp_columns = ['mottatt_tid', 'registrert_tid', 'ferdig_behandlet_tid', 'endret_tid', 'teknisk_tid']
    for col in timestamp_columns:
        df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

    date_columns = ['utbetalt_tid', 'forventet_oppstart_tid']
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors='coerce').apply(lambda x: x.date() if pd.notnull(x) else None)

    df = df.where(pd.notnull(df), None)
    
    user = oracle_secrets['user'] + '[DVH_FAM_HM]'
    dsn_tns = oracledb.makedsn(oracle_secrets['host'], 1521, service_name = oracle_secrets['service'])
    
    with oracledb.connect(user=user, password = oracle_secrets['password'], dsn=dsn_tns) as conn:
        with conn.cursor() as cursor:

            if len(df) > 0:
                rows = [tuple(x) for x in df.values]
                cursor.executemany('''INSERT INTO FAM_HELT_FAGSAK (sekvensnummer, behandling_id, behandling_uuid,relatert_behandling_id, 
                                        relatert_fagsystem, sak_id, saksnummer,aktor_id, mottatt_tid, registrert_tid, ferdig_behandlet_tid, 
                                        utbetalt_tid, endret_tid, forventet_oppstart_tid, teknisk_tid,sak_ytelse, sak_utland, behandling_type, 
                                        behandling_status,behandling_resultat, resultat_begrunnelse, behandling_metode,
                                        behandling_aarsak, opprettet_av, saksbehandler,ansvarlig_beslutter, ansvarlig_enhet, hovudsaka) 
                                         VALUES (:1,:2,:3,:4,:5,:6,:7,:8,TO_TIMESTAMP(:9, 'yyyy-mm-dd HH24:MI:SS.FF6'),
                                         TO_TIMESTAMP(:10, 'yyyy-mm-dd HH24:MI:SS.FF6'),TO_TIMESTAMP(:11, 'yyyy-mm-dd HH24:MI:SS.FF6'),
                                         :12,TO_TIMESTAMP(:13, 'yyyy-mm-dd HH24:MI:SS.FF6'),:14,TO_TIMESTAMP(:15, 'yyyy-mm-dd HH24:MI:SS.FF6'),
                                         :16,:17,:18,:19,:20,:21,:22,:23,:24,:25,:26,:27,:28)''',rows)
                conn.commit()

                convert_aktor_id_query = """
                    
                
                """

In [ ]:
send_data()

In [ ]:
print('Hello, world!')